In [1]:
import sqlite3
import pandas as pd
import numpy as np
import joblib
from datetime import datetime, timedelta

conn = sqlite3.connect('../apex.db')
cursor = conn.cursor()

cursor.executescript('''
DROP TABLE IF EXISTS raw_inspections;
DROP TABLE IF EXISTS dmaic_results;
DROP TABLE IF EXISTS ml_quality_scores;
DROP TABLE IF EXISTS equipment_alerts;

CREATE TABLE raw_inspections (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    piece_id INTEGER,
    process TEXT,
    defect_type TEXT,
    conformite INTEGER
);

CREATE TABLE dmaic_results (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    kpi_name TEXT,
    value REAL,
    target REAL,
    ucl REAL,
    lcl REAL,
    status TEXT,
    sigma_level REAL
);

CREATE TABLE ml_quality_scores (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    piece_id INTEGER,
    process TEXT,
    defect_score REAL,
    defect_type_pred TEXT,
    gonogo_score REAL,
    alert_level TEXT
);

CREATE TABLE equipment_alerts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    equipment_id INTEGER,
    failure_score REAL,
    failure_type_pred TEXT,
    alert_level TEXT
);
''')

conn.commit()
print("Base et tables créées.")

Base et tables créées.


In [2]:
from ucimlrepo import fetch_ucirepo

steel = fetch_ucirepo(id=198)
X_steel = steel.data.features
y_steel = steel.data.targets.idxmax(axis=1)

base_time = datetime(2026, 6, 1)

rows = []
for i in range(len(X_steel)):
    rows.append((
        (base_time + timedelta(minutes=i*5)).isoformat(),
        i,
        'emboutissage',
        y_steel.iloc[i],
        0  # toute pièce avec un défaut détecté = non conforme
    ))

cursor.executemany(
    'INSERT INTO raw_inspections (timestamp, piece_id, process, defect_type, conformite) VALUES (?, ?, ?, ?, ?)',
    rows
)
conn.commit()
print(f"{len(rows)} inspections insérées.")

1941 inspections insérées.


In [3]:
# --- Module A : recharger le modèle et rejouer les prédictions sur Steel Plates ---
model_a = joblib.load('../models/module_a_defect_classifier.pkl')
scaler_a = joblib.load('../models/module_a_scaler.pkl')

X_steel_scaled = scaler_a.transform(X_steel)
proba_a = model_a.predict_proba(X_steel_scaled)
pred_a = model_a.predict(X_steel_scaled)
defect_scores = proba_a.max(axis=1)  # score de confiance de la classe prédite

rows_ml = []
for i in range(len(X_steel)):
    score = defect_scores[i]
    alert = 'critical' if score > 0.85 else ('warning' if score > 0.6 else 'ok')
    rows_ml.append((
        (base_time + timedelta(minutes=i*5)).isoformat(),
        i,
        'emboutissage',
        float(score),
        pred_a[i],
        None,
        alert
    ))

cursor.executemany(
    '''INSERT INTO ml_quality_scores 
       (timestamp, piece_id, process, defect_score, defect_type_pred, gonogo_score, alert_level) 
       VALUES (?, ?, ?, ?, ?, ?, ?)''',
    rows_ml
)
conn.commit()
print(f"{len(rows_ml)} scores Module A insérés.")

1941 scores Module A insérés.


In [20]:
model_b = joblib.load('../models/module_b_gonogo_classifier.pkl')
top_features_b = joblib.load('../models/module_b_selected_features.pkl')
threshold_b = joblib.load('../models/module_b_optimal_threshold.pkl')

# On charge uniquement les colonnes nécessaires (les 80 features + Id), 
# directement en lisant un échantillon du CSV — beaucoup plus rapide que tout recharger
cols_to_load = ['Id'] + top_features_b

sample_chunks = []
for chunk in pd.read_csv('../data/bosch_subset/bosch-production-line-performance/train_numeric.csv/train_numeric.csv', 
                          usecols=cols_to_load, chunksize=50000):
    sample_chunks.append(chunk.sample(frac=0.02, random_state=42))

df_sample_b = pd.concat(sample_chunks, ignore_index=True)
df_sample_b = df_sample_b.fillna(0)

X_test_b_reduced = df_sample_b[top_features_b]

print(X_test_b_reduced.shape)

(23675, 80)


In [21]:
sample_b = X_test_b_reduced.sample(n=2000, random_state=42)
proba_b = model_b.predict_proba(sample_b)[:, 1]

rows_gonogo = []
for idx, (i, score) in enumerate(zip(sample_b.index, proba_b)):
    alert = 'critical' if score > threshold_b else ('warning' if score > threshold_b*0.6 else 'ok')
    rows_gonogo.append((
        (base_time + timedelta(minutes=idx*3)).isoformat(),
        int(i),
        'assemblage',
        None,
        None,
        float(score),
        alert
    ))

cursor.executemany(
    '''INSERT INTO ml_quality_scores 
       (timestamp, piece_id, process, defect_score, defect_type_pred, gonogo_score, alert_level) 
       VALUES (?, ?, ?, ?, ?, ?, ?)''',
    rows_gonogo
)
conn.commit()
print(f"{len(rows_gonogo)} scores Module B insérés.")

2000 scores Module B insérés.


In [22]:
model_c = joblib.load('../models/module_c_equipment_classifier.pkl')
scaler_c = joblib.load('../models/module_c_scaler.pkl')
le_type_c = joblib.load('../models/module_c_type_encoder.pkl')

df_ai4i_db = pd.read_csv('../data/ai4i+2020+predictive+maintenance+dataset/ai4i2020.csv')
df_ai4i_db['Type_encoded'] = le_type_c.transform(df_ai4i_db['Type'])

feature_cols_c = ['Air temperature [K]', 'Process temperature [K]',
                   'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_encoded']

X_ai4i_scaled = scaler_c.transform(df_ai4i_db[feature_cols_c])
proba_c = model_c.predict_proba(X_ai4i_scaled)[:, 1]
pred_c = model_c.predict(X_ai4i_scaled)

# Déterminer le type de panne prédit (le sous-type actif le plus probable, ou "None")
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

rows_equip = []
for i in range(len(df_ai4i_db)):
    score = proba_c[i]
    alert = 'critical' if score > 0.7 else ('warning' if score > 0.4 else 'ok')
    
    # sous-type réel associé (à titre indicatif, basé sur les données, pas une prédiction du modèle)
    active_types = [col for col in failure_cols if df_ai4i_db.iloc[i][col] == 1]
    failure_type = active_types[0] if active_types else None
    
    rows_equip.append((
        (base_time + timedelta(minutes=i*10)).isoformat(),
        int(df_ai4i_db.iloc[i]['UDI']),
        float(score),
        failure_type,
        alert
    ))

cursor.executemany(
    '''INSERT INTO equipment_alerts 
       (timestamp, equipment_id, failure_score, failure_type_pred, alert_level) 
       VALUES (?, ?, ?, ?, ?)''',
    rows_equip
)
conn.commit()
print(f"{len(rows_equip)} alertes équipement insérées.")

10000 alertes équipement insérées.


In [25]:
from scipy.stats import norm

# --- PPM sur Steel Plates (process emboutissage) ---
total_pieces = len(X_steel)
total_defauts = total_pieces

ppm = (total_defauts / total_pieces) * 1_000_000
ftq = (1 - total_defauts / total_pieces) * 100

# --- DPMO et niveau sigma sur Bosch (process assemblage) ---
# Valeur reprise directement du Module B (118375 pièces, 690 défauts, calculé précédemment)
taux_defaut_bosch = 690 / 118375
dpmo_bosch = taux_defaut_bosch * 1_000_000
sigma_bosch = norm.ppf(1 - taux_defaut_bosch) + 1.5

# --- Taux de panne équipement (Module C) ---
taux_panne_equip = df_ai4i_db['Machine failure'].mean() * 100

kpi_rows = [
    (datetime.now().isoformat(), 'PPM_emboutissage', ppm, 500, None, None, 
     'hors_cible' if ppm > 500 else 'ok', None),
    (datetime.now().isoformat(), 'DPMO_assemblage', dpmo_bosch, 500, None, None,
     'hors_cible' if dpmo_bosch > 500 else 'ok', sigma_bosch),
    (datetime.now().isoformat(), 'Taux_panne_equipement', taux_panne_equip, 2.0, None, None,
     'hors_cible' if taux_panne_equip > 2.0 else 'ok', None),
]

cursor.executemany(
    '''INSERT INTO dmaic_results 
       (timestamp, kpi_name, value, target, ucl, lcl, status, sigma_level) 
       VALUES (?, ?, ?, ?, ?, ?, ?, ?)''',
    kpi_rows
)
conn.commit()
print("KPIs DMAIC insérés :")
for row in kpi_rows:
    print(f"  {row[1]} = {row[2]:.2f} (target: {row[3]}, status: {row[6]})")

KPIs DMAIC insérés :
  PPM_emboutissage = 1000000.00 (target: 500, status: hors_cible)
  DPMO_assemblage = 5828.93 (target: 500, status: hors_cible)
  Taux_panne_equipement = 3.39 (target: 2.0, status: hors_cible)


In [26]:
tables = ['raw_inspections', 'dmaic_results', 'ml_quality_scores', 'equipment_alerts']

for table in tables:
    cursor.execute(f'SELECT COUNT(*) FROM {table}')
    count = cursor.fetchone()[0]
    print(f"{table}: {count} lignes")

conn.close()
print("\nConnexion fermée.")

raw_inspections: 1941 lignes
dmaic_results: 3 lignes
ml_quality_scores: 7941 lignes
equipment_alerts: 20000 lignes

Connexion fermée.
